In [4]:
import pandas as pd
import re

def remove_numbers(text):
    # Remove numbers (digits) from the text
    return re.sub(r'\d+', '', text)

# Function to remove months from text
def remove_months(text):
    return re.sub(month_pattern, '', text, flags=re.IGNORECASE)


df_gender = pd.read_csv('Data/gender_Apr25-1.csv')
df_health = pd.read_csv('Data/healthworkers_Apr18.csv')
df_journalists = pd.read_csv('Data/journalists_Apr18.csv')
df_peacekeeping = pd.read_excel('Data/peacekeeping_2025-05-02.xlsx')

df_gender = df_gender[['notes']]
df_gender[['target']] = 'gender'

df_health = df_health[['notes']]
df_health[['target']] = 'health' 

df_journalists = df_journalists[['notes']]
df_journalists[['target']] = 'journalists' 


df_peacekeeping[['notes']] = df_peacekeeping[['NOTES']]
df_peacekeeping = df_peacekeeping[['notes']]
df_peacekeeping[['target']] = 'peacekepers'

df_merged = pd.concat([df_gender, df_health, df_journalists, df_peacekeeping], ignore_index=True) 
df_merged['clean_text'] = df_merged['notes'].apply(remove_numbers)

months = [
    "january", "february", "march", "april", "may", "june", 
    "july", "august", "september", "october", "november", "december"
]


# Create a regex pattern to match month names (case insensitive)
month_pattern = r'\b(?:' + '|'.join(months) + r')\b'
# Apply the function to the 'text' column
df_merged['clean_text'] = df_merged['clean_text'].apply(remove_months)

df_merged

,notes,target,clean_text
0,"On 25 April 2025, in Capilla del Monte (Cordob...",gender,"On , in Capilla del Monte (Cordoba), a large..."
1,"Around 25 April 2025 (as reported), in Salvado...",gender,"Around (as reported), in Salvador - Suburbi..."
2,"On 25 April 2025, about 200 Israelis from the ...",gender,"On , about Israelis from the Shift movemen..."
3,"On 25 April 2025, in Leon de los Aldama, Guana...",gender,"On , in Leon de los Aldama, Guanajuato, A wo..."
4,"On 25 April 2025, in Sabanas de Xalostoc, Vera...",gender,"On , in Sabanas de Xalostoc, Veracruz, armed..."
...,...,...,...
152120,Houthi forces reportedly fired a Katyusha rock...,peacekepers,Houthi forces reportedly fired a Katyusha rock...
152121,"On 2 March 2019, anti-Houthi forces reportedly...",peacekepers,"On , anti-Houthi forces reportedly opened fi..."
152122,"On 26 February 2019, pro-Houthi forces shelled...",peacekepers,"On , pro-Houthi forces shelled the UNMHA con..."
152123,Houthi forces reportedly fired at the UN team ...,peacekepers,Houthi forces reportedly fired at the UN team ...


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
import re

KeyError: 'label'

In [10]:
from sklearn.preprocessing import LabelEncoder

tokenizer = Tokenizer(num_words=5000)  # Limit the vocabulary size to 5000 words
tokenizer.fit_on_texts(df_merged['clean_text'])
X = tokenizer.texts_to_sequences(df_merged['clean_text'])

label_encoder = LabelEncoder()
y_int = label_encoder.fit_transform(df_merged['target'])

X_pad = pad_sequences(X, padding='post', maxlen=10)
y = to_categorical(y_int, num_classes=4)

In [11]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X_pad, y, test_size=0.2, random_state=42, stratify=y)

# Define the model
model = Sequential([
    Embedding(input_dim=5000, output_dim=64, input_length=10),
    GlobalAveragePooling1D(),
    Dense(32, activation='relu'),
    Dense(4, activation='softmax')  # Assuming 4 target classes
])

# Compile the model
model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, 
                    epochs=10, 
                    batch_size=32, 
                    validation_data=(X_test, y_test))

Epoch 1/10


/Users/felipe_q/Desktop/AI_models/MACHINE_LEARNING/mlenv/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


3804/3804 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 0.7711 - loss: 0.6046 - val_accuracy: 0.8378 - val_loss: 0.4368
Epoch 2/10
3804/3804 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - accuracy: 0.8526 - loss: 0.3951 - val_accuracy: 0.8392 - val_loss: 0.4336
Epoch 3/10
3804/3804 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - accuracy: 0.8572 - loss: 0.3780 - val_accuracy: 0.8395 - val_loss: 0.4327
Epoch 4/10
3804/3804 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - accuracy: 0.8613 - loss: 0.3630 - val_accuracy: 0.8394 - val_loss: 0.4344
Epoch 5/10
3804/3804 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 0.8654 - loss: 0.3522 - val_accuracy: 0.8397 - val_loss: 0.4366
Epoch 6/10
3804/3804 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - accuracy: 0.8692 - loss: 0.3398 - val_accuracy: 0.8418 - val_loss: 0.4425
Epoch 7/10
3804/3804 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 0.8737 - loss: 0.3276 - val_accuracy: 0.8371 - val_loss: 0.4494
Epoch 8/10
3804/3804 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 0.8780 - loss: 0.3198 - val_

In [12]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc:.4f}")

951/951 ━━━━━━━━━━━━━━━━━━━━ 1s 583us/step - accuracy: 0.8333 - loss: 0.4813
Test Accuracy: 0.8346
